# Presentation figures

Every figure for the presentation, in one consistent style. Style, the example run (`pf.EXAMPLE`) and the
pipeline parameters (`pf.PUPIL_PARAMS`, `pf.LOCOM_PARAMS`) live in [presentation_figures.py](../presentation_figures.py) (`pf`); each cell
below draws one figure, saves it to `figures/` at the repo root (PNG 300 dpi + SVG; change the folder with `PO2_FIG_DIR` in `.env`) and shows it.

To change *what* a figure shows, edit the matching `fig_*` function or the parameter sets in [presentation_figures.py](../presentation_figures.py). Thanks to autoreload, the next run of the cell picks up the edit. The same figures can be rebuilt without Jupyter by running `python presentation_figures.py` in a terminal (with the `po2` environment active).

## Before you run this

> **New to Python or Jupyter?** Do the one-time setup in the [README](../README.md#full-setup-from-zero) first: install conda, create the `po2` environment, put the data in `data/`.

1. **Choose the kernel.** Top-right of this notebook in VS Code, click *Select Kernel* → *Python Environments* → **po2**. (Jupyter Lab: *Kernel → Change Kernel → Python (po2)*.) The kernel is the Python process that runs the cells. Picking `po2` makes it use the packages from `environment.yml`.
2. **Run cells top to bottom.** Click in a cell and press **Shift + Enter** to run it and move to the next one, or use **Run All**. Cells share variables like the MATLAB workspace, so a cell only works once the cells above it have run.
3. **Changed a parameter?** Re-run that cell and every cell below it. If things get confusing, **Restart** the kernel (it clears all variables, like `clear all`) and run everything again.
4. Figures and tables appear under the cell that made them. Text printed in red/pink is usually a *warning* (it keeps going), not an *error* (it stops, with a "Traceback" ending in the error message).

In [ ]:
# % lines are Jupyter commands: reload edited .py files automatically, draw figures as static images
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import presentation_figures as pf   # this repo's figure module (style, parameter sets, one fig_* function per figure)

print("Saving figures to:", pf.FIG_DIR)
print("Example:", pf.example_label())
print("Pupil:", pf.PUPIL_PARAMS.describe())
print("Locomotion:", pf.LOCOM_PARAMS.describe())
print("Sleep state:", pf.SLEEP_STATE_PARAMS.describe())

## 1. Raw and filtered pupil
Raw pupil (gray) and the filtered pupil used for event detection (moving median, then zero-phase low-pass).

In [ ]:
fig = pf.fig_pupil_raw_filtered()
pf.save(fig, "pupil_raw_filtered")
plt.show()

## 2. Pupil-dilation event detection
Top: pupil with each event's extraction window (shaded) and event number. Bottom: dPupil/dt with the threshold and detected peaks.

In [ ]:
fig = pf.fig_pupil_event_detection()
pf.save(fig, "pupil_event_detection")
plt.show()

## 3. Single event: pupil and pO2
One event (`event_number`, 1-based, as numbered in figure 2) over its extraction window.
Top: filtered pupil and dPupil/dt (two y-axes) with the threshold and the event peak. Bottom: raw pO2 of every point (gray) and their mean (black).

In [ ]:
event_number = 1
fig = pf.fig_event_pupil_po2(event_number=event_number)
pf.save(fig, f"event{event_number}_pupil_po2")
plt.show()

## 4. Responses by event AUC bin (all runs)
Every run in `pf.ALL_RUNS`. Events are split into percentile bins of `metric` (`pt.bin_labels`); each event window is
interpolated onto a `pf.DT_S` grid (`pt.event_responses`) and averaged per bin (`pt.group_average`).
Top: behavior signal (defaults to the event method's filtered signal). Middle: raw pO2 (each event averaged over its points first).
Bottom: histogram of `metric`, colored by bin. For another behavior, pass `params` with a different `event_method`.

In [ ]:
fig = pf.fig_binned_event_responses(metric="event_auc", bins=pf.AUC_BINS, po2_signal="po2")
pf.save(fig, "pupil_po2_by_auc")
plt.show()

## 5. Responses by locomotion peak height (all runs)
Same figure as section 4 with locomotion events (`pf.LOCOM_PARAMS`): events binned by `event_height`, the peak of the
smoothed accelerometer signal. Top: smoothed locomotion. Middle: raw pO2. Bottom: peak-height histogram, colored by bin.

In [ ]:
fig = pf.fig_binned_event_responses(params=pf.LOCOM_PARAMS, metric="event_height", bins=pf.TERTILE_BINS,
                                    title_text="Locomotion and pO$_2$ by locomotion peak height")
pf.save(fig, "locom_po2_by_height")
plt.show()

## 6. Responses by sleep-state transition (all runs)
Same figure with sleep-state events (`pf.SLEEP_STATE_PARAMS`): peaks of dSleepState/dt (positive and negative), binned by
value of `event_height`: −1 = NREM → awake, +1 = awake → NREM (`pf.SLEEP_TRANSITION_BINS`).
Top: sleep state. Middle: raw pO2. Bottom: count per transition type.

In [ ]:
fig = pf.fig_binned_event_responses(params=pf.SLEEP_STATE_PARAMS, metric="event_height",
                                    bins=pf.SLEEP_TRANSITION_BINS, percentile=False,
                                    bin_names=pf.SLEEP_TRANSITION_NAMES,
                                    title_text="Sleep state and pO$_2$ by transition type")
pf.save(fig, "sleep_po2_by_transition")
plt.show()